# Text-to-Image Studio AI 2.0: Google Colab launcher
Run the cells **in order**. Before starting: **Runtime → Change runtime type → GPU (T4 is enough)**.

## 1. GPU and CUDA check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found (no GPU runtime?)")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CUDA build:", torch.version.cuda)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    gb = p.total_memory / 1024**3
    print(f"GPU: {p.name} | {gb:.1f} GB | compute capability {p.major}.{p.minor}")
    if gb >= 22:   print("Profile: high-end. SDXL 1024x1024, batch up to 4, LoRA training OK.")
    elif gb >= 14: print("Profile: T4-class. SDXL works (30-60 s/image); SD 1.5 is fast; LoRA training with batch size 1.")
    elif gb >= 8:  print("Profile: mid-range. Prefer SD 1.5; SDXL only with CPU offload.")
    else:          print("Profile: low VRAM. Use SD 1.5 at 512x512, batch size 1.")
else:
    print("WARNING: no GPU. Switch the runtime type to GPU, otherwise generation and training will be extremely slow.")

## 2. Mount Google Drive (persistent storage)
Images, datasets, checkpoints, adapters and the database go to `MyDrive/TextToImageStudio`, so they survive disconnects.

In [ ]:
import os
USE_DRIVE = True   # set False to keep everything in the temporary Colab disk
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["STUDIO_DATA_DIR"] = "/content/drive/MyDrive/TextToImageStudio"
    # Optional: keep downloaded model weights on Drive too (saves re-downloading after a disconnect)
    # os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
else:
    os.environ["STUDIO_DATA_DIR"] = "/content/studio_data"
os.makedirs(os.environ["STUDIO_DATA_DIR"], exist_ok=True)
print("Data directory:", os.environ["STUDIO_DATA_DIR"])

## 3. Get the project
Choose **one**: a ZIP you uploaded to Drive, or a GitHub repository.

In [ ]:
import os, shutil, zipfile, subprocess
PROJECT_ZIP = "/content/drive/MyDrive/Text-to-Image-Studio-AI.zip"   # option A: ZIP on Drive
REPO_URL = ""                                                        # option B: e.g. https://github.com/<you>/Text-to-Image-Studio-AI.git
TARGET = "/content/Text-to-Image-Studio-AI"
if os.path.isdir(TARGET) and os.path.exists(f"{TARGET}/app.py"):
    print("Project already present:", TARGET)
elif REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, TARGET], check=True)
elif os.path.exists(PROJECT_ZIP):
    with zipfile.ZipFile(PROJECT_ZIP) as z: z.extractall("/content")
    print("Extracted ZIP to /content")
else:
    raise FileNotFoundError("Upload Text-to-Image-Studio-AI.zip to Drive (or set REPO_URL), then re-run this cell.")
os.chdir(TARGET)
print(os.listdir("."))

## 4. Install dependencies
Colab already has PyTorch, so it is **not** reinstalled.

In [ ]:
!pip install -q -r requirements.txt
# Colab ships an old optional 'torchao' that newer peft rejects when a LoRA adapter is attached; this project does not use it.
!pip uninstall -y -q torchao


## 5. (Optional) Pre-download models
Saves time later. SD 1.5 is about 4 GB, SDXL about 7 GB, CLIP ViT-B/32 about 0.6 GB. Skip what you do not need.

In [ ]:
DOWNLOAD_SD15, DOWNLOAD_SDXL, DOWNLOAD_CLIP = True, False, True
from huggingface_hub import snapshot_download
if DOWNLOAD_SD15:
    snapshot_download("stable-diffusion-v1-5/stable-diffusion-v1-5", allow_patterns=["*.json","*.txt","*fp16.safetensors","tokenizer/*","scheduler/*"])
if DOWNLOAD_SDXL:
    snapshot_download("stabilityai/stable-diffusion-xl-base-1.0", allow_patterns=["*.json","*.txt","*fp16.safetensors","tokenizer*/*","scheduler/*"])
if DOWNLOAD_CLIP:
    snapshot_download("openai/clip-vit-base-patch32")
print("Model cache ready.")

## 6. Run the unit tests (optional)

In [ ]:
!python -m unittest discover -s tests 2>&1 | tail -n 15

## 7. Launch the app
Starts Streamlit and opens a public Cloudflare tunnel. **Open the printed link.**

In [ ]:
import subprocess, time, re, os, urllib.request, stat
os.chdir("/content/Text-to-Image-Studio-AI")
subprocess.run(["pkill", "-f", "streamlit run"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)

log = open("streamlit.log", "w")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"], stdout=log, stderr=subprocess.STDOUT)
time.sleep(6)

CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, os.stat(CF).st_mode | stat.S_IEXEC)
tunnel = subprocess.Popen([CF, "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 60 and url is None:
    line = tunnel.stderr.readline()
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
    if m: url = m.group(0)
print("OPEN THIS LINK:", url or "no link found, see the troubleshooting cell below")

## Troubleshooting
* No link: run the cell below to see the Streamlit log, then re-run cell 7. Alternative tunnel: `!npx localtunnel --port 8501` (the password is the output of `!curl ipv4.icanhazip.com`).
* Out of memory: Settings → Performance → CPU offload = always, or use SD 1.5.
* After a disconnect: re-run cells 2, 3, 4 and 7; your data is on Drive.

In [ ]:
!tail -n 30 /content/Text-to-Image-Studio-AI/streamlit.log
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv